In [3]:
# Task 3.1

import sqlite3

table = """
CREATE TABLE IF NOT EXISTS Board(
    xCoordinate INTEGER CHECK (1 <= xCoordinate <= 4),
    yCoordinate INTEGER CHECK (1 <= yCoordinate <= 4),
    hasMine INTEGER CHECK (hasMine in (0,1)),
    checked INTEGER CHECK (checked in (0,1))
)
"""

with sqlite3.connect("minesweeper.db") as con:
    c = con.cursor()

    c.execute(table)
    con.commit()


In [3]:
# Task 3.2

import sqlite3

def generate_data():

    import random
    mines = [[1,0] for _ in range(10)] + [[0,0] for _ in range(6)]
    random.shuffle(mines)

    coords = [[x,y] for x in range(1,5) for y in range(1,5)]

    board = [coords[i] + mines[i] for i in range(16)]

    return board


def get_no_mines(x,y,board):
    surr_mines = 0
    surr_coords = [[x+i,y+j] 
                   for j in range(-1,2) 
                   for i in range(-1,2)
                    if (0 < x+i < 5) and (0 < y+j < 5)
                ]
    
    for i in surr_coords:
        x,y = i
        if board[y-1][x-1][2]:
            surr_mines += 1

    return surr_mines



with sqlite3.connect("minesweeper.db") as con:
    c = con.cursor()
    c.execute("DELETE FROM Board")
    data = generate_data()
    c.executemany("INSERT INTO Board VALUES (?,?,?,?)",data)
    con.commit()


from socket import socket

s = socket()

port = 6789
s.bind(('',port))

s.listen(1)

print("-------------------------")
print("MINESWEEPER GAME [SERVER]")
print("-------------------------")
print()

no_of_moves = 0
c,addr = s.accept()
board = generate_data()
board = list(zip(*[board[(i*4):(i*4)+4] for i in range(4)]))

while True:
    print(*board, sep="\n")
    data = c.recv(4096).decode()
    x,y = [int(i) for i in data.split(",",maxsplit=1)]
    print(f"User x coordinate: {x}, User y coordinate: {y}")
    no_of_moves += 1
    if no_of_moves == 6:
        break
    with sqlite3.connect("minesweeper.db") as con:
        cursor = con.cursor()
        cursor.execute("UPDATE Board SET checked=1 WHERE xCoordinate = ? AND yCoordinate = ?",(x,y))
    print(board[y-1][x-1])
    if board[y-1][x-1][2]:
        print("Hit MINE")
        c.send("MINE\n".encode())
        break
    else:
        no_of_mines = get_no_mines(x,y,board)
        print(f"no_of_surrounding_mines: {no_of_mines}")
        print(f"no_of_user_moves_made: {no_of_moves}")
        c.sendall(f"{no_of_mines},{no_of_moves}\n".encode())
              








-------------------------
MINESWEEPER GAME [SERVER]
-------------------------

([1, 1, 1, 0], [2, 1, 1, 0], [3, 1, 0, 0], [4, 1, 0, 0])
([1, 2, 0, 0], [2, 2, 0, 0], [3, 2, 1, 0], [4, 2, 0, 0])
([1, 3, 1, 0], [2, 3, 1, 0], [3, 3, 0, 0], [4, 3, 1, 0])
([1, 4, 1, 0], [2, 4, 1, 0], [3, 4, 1, 0], [4, 4, 1, 0])
User x coordinate: 2, User y coordinate: 2
[2, 2, 0, 0]
no_of_surrounding_mines: 5
no_of_user_moves_made: 1
([1, 1, 1, 0], [2, 1, 1, 0], [3, 1, 0, 0], [4, 1, 0, 0])
([1, 2, 0, 0], [2, 2, 0, 0], [3, 2, 1, 0], [4, 2, 0, 0])
([1, 3, 1, 0], [2, 3, 1, 0], [3, 3, 0, 0], [4, 3, 1, 0])
([1, 4, 1, 0], [2, 4, 1, 0], [3, 4, 1, 0], [4, 4, 1, 0])


ValueError: invalid literal for int() with base 10: ''